# Atelier ML - Partie 1 : préparation des données

Objectif : préparer les données du Pokédex et des combats pour pouvoir prédire le vainqueur d'un combat de Pokémons.

## Modules utilisés

- **Pandas** : manipulation et analyse de données

In [ ]:
import pandas as pd

pd.set_option('display.max_columns', None)  # Affiche toutes les colonnes
# pd.set_option('future.no_silent_downcasting', True)  # option du cours, obsolète avec pandas 3

## De quelles données disposons-nous ?

On charge le Pokédex dans un Dataframe avec `read_csv`, puis on affiche le nom des colonnes (features).

In [ ]:
nosPokemons = pd.read_csv("data/pokedex.csv")

# Affichage des colonnes du Dataframe
print(nosPokemons.columns.values)

## Affichage des dix premières lignes

Chaque ligne est une **observation** (un Pokémon) et chaque colonne une **feature** (caractéristique).

In [ ]:
print(nosPokemons.head(10))

## Quelles sont les features de catégorisation ?

Elles permettent de classer les Pokémons dans des groupes :

- **Catégorielles** : `TYPE_1`, `TYPE_2`, `LEGENDAIRE`
- **Ordinale** : `NOMBRE_GENERATIONS` (la génération, une donnée chiffrée)

Ces features ne devront pas être supprimées lors du nettoyage.

## Quelles données sont de type numérique ?

`NUMERO`, `POINTS_DE_VIE`, `NIVEAU_ATTAQUE`, `NIVEAU_DEFENSE`, `NIVEAU_ATTAQUE_SPECIALE`, `NIVEAU_DEFENSE_SPECIALE` et `VITESSE` sont des données **numériques discrètes** (des valeurs entières qu'on peut compter).

## Que faut-il penser de la feature LEGENDAIRE ?

`LEGENDAIRE` contient le texte `VRAI` ou `FAUX`. Le Machine Learning n'utilise que des données numériques : on la transforme donc en `1` (VRAI) ou `0` (FAUX).

In [ ]:
# remplacement de VRAI FAUX par 1 0
nosPokemonsClean = nosPokemons
nosPokemonsClean["LEGENDAIRE"] = nosPokemonsClean["LEGENDAIRE"].replace("VRAI", 1)
nosPokemonsClean["LEGENDAIRE"] = nosPokemonsClean["LEGENDAIRE"].replace("FAUX", 0)
nosPokemonsClean["LEGENDAIRE"] = nosPokemonsClean["LEGENDAIRE"].astype(int)
print(nosPokemonsClean.head())

## Manque-t-il des données ?

On compte le nombre d'observations et de features avec `shape`.

In [ ]:
# Comptage du nombre d'observations et de features
print(nosPokemonsClean.shape)

La fonction `info()` donne le nombre de valeurs non vides par feature.

In [ ]:
# Informations sur notre jeu de données
print(nosPokemonsClean.info())

Constats :

- 800 observations et 12 features
- il manque **1 nom** (799 sur 800)
- il manque 387 `TYPE_2` : c'est normal, un Pokémon peut ne pas avoir de type secondaire

## À la recherche des features manquantes

On cherche l'observation dont le nom est vide (`isnull`).

In [ ]:
print(nosPokemonsClean[nosPokemonsClean['NOM'].isnull()])

C'est l'observation 62. Les Pokémons sont dans l'ordre du Pokédex : on regarde le Pokémon avant et après pour retrouver le nom manquant dans le [Pokédex](https://www.pokepedia.fr/Liste_des_Pok%C3%A9mon_dans_l%27ordre_du_Pok%C3%A9dex_National).

In [ ]:
print(nosPokemonsClean['NOM'][61])
print(nosPokemonsClean['NOM'][63])

Entre Ferosinge et Caninos, le Pokédex indique **Colossinge**.

Le cours utilise `nosPokemonsClean['NOM'][62] = "Colossinge"`, mais avec pandas 3 cette écriture ne modifie plus le Dataframe (le nom reste vide). On utilise `.loc[ligne, colonne]` à la place.

In [ ]:
nosPokemonsClean.loc[62, 'NOM'] = "Colossinge"
print(nosPokemonsClean['NOM'][62])

Cette modification ne concerne que le Dataframe. Le fichier `data/pokedex.csv` a aussi été corrigé :

- Avant : `63,,Combat,,65,105,60,60,70,95,1,FAUX`
- Après : `63,Colossinge,Combat,,65,105,60,60,70,95,1,FAUX`

## Place aux observations des combats

Même démarche que pour le Pokédex : chargement, features, dix premières observations, nombre d'observations et données manquantes.

In [ ]:
# Chargement des données des combats
combats = pd.read_csv("data/combats.csv")

# Affichage des colonnes du Dataframe
print(combats.columns.values)

# Affichage des 10 premières lignes du Dataframe
print(combats.head(10))

# Comptage du nombre de lignes et de colonnes
print(combats.shape)

# Informations sur notre jeu de données
print(combats.info())

Constats :

- 50 000 observations (combats) et 3 features : `Premier_Pokemon`, `Second_Pokemon` et `Pokemon_Gagnant`
- `Pokemon_Gagnant` contient le numéro du Pokémon qui a gagné le combat
- toutes les features sont numériques
- il ne manque aucune donnée

## Assemblage des observations

On veut savoir, pour chaque Pokémon, le nombre de combats menés et le nombre de victoires.

### Nombre de combats menés

Un Pokémon peut combattre en première ou en seconde position. Son nombre de combats = nombre de fois en première position + nombre de fois en seconde position.

`groupby()` regroupe les combats par Pokémon et `count()` compte les combats de chaque groupe.

In [ ]:
nbFoisPremierePosition = combats.groupby('Premier_Pokemon').count()
print(nbFoisPremierePosition)

La première colonne est le critère de regroupement (le numéro du Pokémon). Les deux autres colonnes contiennent la même valeur : le nombre de fois où le Pokémon a combattu en première position.

In [ ]:
nbFoisSecondePosition = combats.groupby('Second_Pokemon').count()
print(nbFoisSecondePosition)

On additionne les deux pour obtenir le nombre total de combats. Avec Pandas, un simple `+` suffit pour additionner les deux tableaux ligne par ligne.

Ne pas tenir compte des noms de colonnes affichés : la colonne qui contient une valeur donne le nombre total de combats (ex. Pokémon 1 : 70 + 63 = 133). Les colonnes `NaN` viennent du fait que les deux tableaux n'ont pas exactement les mêmes noms de colonnes.

In [ ]:
nombreTotalDeCombats = nbFoisPremierePosition + nbFoisSecondePosition
print(nombreTotalDeCombats)

### Nombre de combats gagnés

Même principe en regroupant par `Pokemon_Gagnant`.

In [ ]:
nombreDeVictoires = combats.groupby('Pokemon_Gagnant').count()
print(nombreDeVictoires)

In [ ]:
print(nombreDeVictoires.info())

Seulement **783** Pokémons sur 800 apparaissent dans les victoires : 17 Pokémons n'ont donc jamais gagné de combat.